Ten skrypt ładuje dokumentację Hugging Face, tworzy z niej bazę wektorową FAISS przy użyciu LangChain i definiuje narzędzie do wyszukiwania semantycznego w tej bazie. Następnie skrypt porównuje dwie metody odpowiadania na pytania: standardowy RAG i agentowy RAG wykorzystujący stworzone narzędzie.

# Setup

In [ ]:
!pip install -qU langchain langchain-community sentence-transformers smolagents datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

*   `pandas`: Biblioteka do analizy i manipulacji danymi, oferująca struktury danych takie jak DataFrame.
*   `langchain`: Framework do tworzenia aplikacji wykorzystujących modele językowe (LLM). Ułatwia łączenie LLM z różnymi źródłami danych i narzędziami.
*   `langchain-community`: Zbiór integracji społecznościowych dla LangChain, rozszerzający jego możliwości o dostęp do różnych modeli, wektorowych baz danych i innych zasobów.
*   `sentence-transformers`: Biblioteka do generowania embeddingów zdań (wektorowych reprezentacji tekstu). Embeddingi te są używane do porównywania podobieństwa semantycznego między tekstami.
*   `faiss-cpu`:  Biblioteka Facebook AI Similarity Search, wersja przeznaczona do działania na procesorze (CPU). Służy do szybkiego wyszukiwania wektorów podobnych w dużych zbiorach danych.
*   `smolagents`: Biblioteka do tworzenia agentów opartych na LLM, umożliwiająca im interakcję z otoczeniem i wykonywanie zadań.
*   `datasets`:  Biblioteka od Hugging Face zapewniająca łatwy dostęp do wielu publicznych zbiorów danych używanych w uczeniu maszynowym.

In [ ]:
# Third-party data handling
from tqdm import tqdm

# HuggingFace ecosystem
import datasets
from huggingface_hub import InferenceClient, login
from transformers import AutoTokenizer

# LangChain components
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import VectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

# SmolaGents
from smolagents import Tool, HfApiModel, ToolCallingAgent

try:
    from google.colab import userdata
except ModuleNotFoundError:
    print("This is not google colab env.")

Ten kod to sekcja importów bibliotek Pythona używanych w programie.  Organizuje je tematycznie i zawiera komentarze wyjaśniające przeznaczenie każdej grupy.

*   **HuggingFace ecosystem:**  Importuje biblioteki związane z platformą Hugging Face, która jest popularna w dziedzinie uczenia maszynowego i przetwarzania języka naturalnego:
    *   `datasets`: Do pobierania i zarządzania zbiorami danych.
    *   `huggingface_hub`:  Do interakcji z repozytoriami modeli na platformie Hugging Face, w tym logowania się do konta (funkcja `login`) i pobierania snapshotów modeli (`snapshot_download`).
    *   `transformers`: Do pracy z modelami transformatorowymi, takimi jak BERT czy GPT.  W szczególności importuje `AutoTokenizer`, który automatycznie dobiera odpowiedni tokenizer dla danego modelu.
*   **LangChain components:** Importuje elementy frameworku LangChain:
    *   `Document`: Klasa reprezentująca pojedynczy dokument tekstowy.
    *   `RecursiveCharacterTextSplitter`:  Narzędzie do dzielenia tekstu na mniejsze fragmenty (chunks) w celu lepszego dopasowania do ograniczeń modeli językowych.
    *   `VectorStore`: Interfejs dla baz wektorowych, które służą do przechowywania i wyszukiwania embeddingów.
    *   `HuggingFaceEmbeddings`: Klasa generująca embeddingi tekstu przy użyciu modeli z Hugging Face.
    *   `FAISS`: Implementacja bazy wektorowej FAISS (Facebook AI Similarity Search).
    *   `DistanceStrategy`:  Określa metrykę odległości używaną do porównywania wektorów w bazie wektorowej.
*   **SmolaGents:** Importuje komponenty biblioteki SmolaGents, która służy do tworzenia agentów opartych na modelach językowych:
    *   `Tool`: Klasa reprezentująca narzędzie dostępne dla agenta.
    *   `HfApiModel`:  Klasa opakowująca modele Hugging Face API.
    *   `ToolCallingAgent`: Klasa implementująca agenta, który może wywoływać narzędzia.

In [ ]:
class CFG:
    dataset = "m-ric/huggingface_doc"
    dataset2 = "m-ric/huggingface_doc_qa_eval"
    model1 = "thenlper/gte-small"
    model2 = "meta-llama/Llama-3.1-8B-Instruct"


login(userdata.get("HF_TOKEN"))

# Funkcje

In [ ]:
class RetrieverTool(Tool):
    name = "retriever"
    description = "Using semantic similarity, retrieves some documents from the knowledge base that have the closest embeddings to the input query."
    inputs = {
        "query": {
            "type": "string",
            "description": "The query to perform. This should be semantically close to your target documents. Use the affirmative form rather than a question.",
        }
    }
    output_type = "string"

    def __init__(self, vectordb: VectorStore, **kwargs):
        super().__init__(**kwargs)
        self.vectordb = vectordb

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "Your search query must be a string"

        docs = self.vectordb.similarity_search(
            query,
            k=7,
        )

        return "\nRetrieved documents:\n" + "".join(
            [
                f"===== Document {str(i)} =====\n" + doc.page_content
                for i, doc in enumerate(docs)
            ]
        )

Ten kod definiuje klasę `RetrieverTool`, która reprezentuje narzędzie (tool) do wyszukiwania dokumentów w bazie wektorowej na podstawie podobieństwa semantycznego.

# Dane

In [5]:
knowledge_base = datasets.load_dataset(CFG.dataset, split="train")

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

In [6]:
knowledge_base[0]

{'text': ' Create an Endpoint\n\nAfter your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. \n\n## 1. Enter the Hugging Face Repository ID and your desired endpoint name:\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />\n\n## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_region.png" alt="select region" />\n\n## 3. Defi

In [7]:
source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"].split("/")[1]})
    for doc in knowledge_base
]


In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained(CFG.model1),
    chunk_size=200,
    chunk_overlap=20,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ".", " ", ""],
)

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
# Split docs and keep only unique ones
print("Splitting documents...")
docs_processed = []
unique_texts = {}
for doc in tqdm(source_docs):
    new_docs = text_splitter.split_documents([doc])
    for new_doc in new_docs:
        if new_doc.page_content not in unique_texts:
            unique_texts[new_doc.page_content] = True
            docs_processed.append(new_doc)


Splitting documents...


100%|██████████| 2647/2647 [01:23<00:00, 31.84it/s]


Ten kod dzieli dokumenty na mniejsze fragmenty i usuwa duplikaty, aby przygotować dane do stworzenia bazy wektorowej.

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name=CFG.model1)
vectordb = FAISS.from_documents(
    documents=docs_processed,
    embedding=embedding_model,
    distance_strategy=DistanceStrategy.COSINE,
)

<ipython-input-10-2940feefbf39>:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name = CFG.model1 )


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/68.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/66.7M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ten kod tworzy bazę wektorową (vector database) z przetworzonych dokumentów (`docs_processed`).

*   `embedding_model = HuggingFaceEmbeddings(model_name = CFG.model1 )`: Tworzy obiekt `embedding_model` klasy `HuggingFaceEmbeddings`. Ten obiekt służy do generowania embeddingów (wektorowych reprezentacji) tekstów.
    *   `model_name = CFG.model1`: Określa nazwę modelu językowego, który ma być używany do generowania embeddingów. Nazwa jest pobierana z atrybutu `model1` klasy `CFG` (wcześniej zdefiniowano go jako `"thenlper/gte-small"`).
*   `vectordb = FAISS.from_documents(...)`: Tworzy obiekt `vectordb` klasy `FAISS`.  `FAISS` to biblioteka do szybkiego wyszukiwania podobnych wektorów, a ten kod tworzy bazę danych wektorowej przy użyciu tej biblioteki.
    *   `documents=docs_processed`: Przekazuje listę przetworzonych dokumentów (`docs_processed`) do bazy wektorowej. Każdy dokument zostanie przekształcony na embedding i dodany do bazy danych.
    *   `embedding=embedding_model`: Przekazuje obiekt `embedding_model`, który będzie używany do generowania embeddingów dla każdego dokumentu.
    *   `distance_strategy=DistanceStrategy.COSINE`: Określa strategię obliczania odległości między wektorami w bazie danych. W tym przypadku wybrano strategię kosinusową (cosine similarity), która mierzy kąt między dwoma wektorami i jest często używana do porównywania podobieństwa semantycznego tekstów.

Ten kod tworzy bazę wektorową `FAISS` zawierającą embeddingi wszystkich dokumentów z listy `docs_processed`, generowane za pomocą modelu "thenlper/gte-small". Baza danych jest skonfigurowana do używania strategii kosinusowej do obliczania podobieństwa między wektorami.  Ta baza danych będzie wykorzystywana przez narzędzie wyszukiwania (`RetrieverTool`) do znajdowania dokumentów najbardziej zbliżonych semantycznie do zadanego zapytania.

# Model

In [ ]:
model = HfApiModel(CFG.model2)

retriever_tool = RetrieverTool(vectordb)
agent = ToolCallingAgent(tools=[retriever_tool], model=model)


Ten kod tworzy agenta, który może korzystać z narzędzia do wyszukiwania dokumentów w bazie wektorowej.

# RAGtime

In [12]:
eval_dataset = datasets.load_dataset("m-ric/huggingface_doc_qa_eval", split="train")

README.md:   0%|          | 0.00/893 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/289k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/65 [00:00<?, ? examples/s]

In [13]:
eval_dataset[0]

{'context': ' `tokenizers-linux-x64-musl`\n\nThis is the **x86_64-unknown-linux-musl** binary for `tokenizers`\n',
 'question': 'What architecture is the `tokenizers-linux-x64-musl` binary designed for?\n',
 'answer': 'x86_64-unknown-linux-musl',
 'source_doc': 'huggingface/tokenizers/blob/main/bindings/node/npm/linux-x64-musl/README.md',
 'standalone_score': 5,
 'standalone_eval': 'The question is asking about the specific architecture for which a binary file, named `tokenizers-linux-x64-musl`, is designed. The terms used in the question are technical but do not depend on a specific context to be understood. The question is clear and can be understood by someone familiar with computer architecture and software without additional context.\n\n',
 'relatedness_score': 5,
 'relatedness_eval': 'The context directly specifies the architecture for which the `tokenizers-linux-x64-musl` binary is designed. It states that this is the binary for `tokenizers` intended for the **x86_64-unknown-lin

In [14]:
reader_llm = InferenceClient(CFG.model2)

In [ ]:
# standard RAG
outputs_standard_rag = []

for example in tqdm(eval_dataset):
    question = example["question"]
    context = retriever_tool(question)

    prompt = f"""Given the question and supporting documents below, give a comprehensive answer to the question.
Respond only to the question asked, response should be concise and relevant to the question.
Provide the number of the source document when relevant.
If you cannot find information, do not give up and try calling your retriever again with different arguments!

Question:
{question}

{context}
"""
    messages = [{"role": "user", "content": prompt}]
    answer = reader_llm.chat_completion(messages).choices[0].message.content

    print("=======================================================")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f"True answer: {example['answer']}")

    results_agentic = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_standard_rag.append(results_agentic)

  2%|▏         | 1/65 [00:02<02:34,  2.41s/it]

Question: What architecture is the `tokenizers-linux-x64-musl` binary designed for?

Answer: The `tokenizers-linux-x64-musl` binary is designed for aarch64-unknown-linux-musl architecture. 

Reference: Document 1
True answer: x86_64-unknown-linux-musl


  3%|▎         | 2/65 [00:04<02:36,  2.48s/it]

Question: What is the purpose of the BLIP-Diffusion model?

Answer: The purpose of the BLIP-Diffusion model is to enable zero-shot subject-driven generation and control-guided zero-shot generation (Source: Document 5).
True answer: The BLIP-Diffusion model is designed for controllable text-to-image generation and editing.


  5%|▍         | 3/65 [00:10<03:56,  3.81s/it]

...

Question: What is the name of the open-source library created by Hugging Face to simplify Transformer acceleration?

Answer: The name of the open-source library created by Hugging Face to simplify Transformer acceleration is Optimum.
True answer: Optimum


 98%|█████████▊| 64/65 [04:37<00:03,  3.51s/it]

Question: What parameter is used to ensure that elements in a row have the same height in Gradio?

Answer: The parameter used to ensure that elements in a row have the same height in Gradio is `equal_height`. It should be passed to the `.style()` method of `gr.Row()`.
True answer: equal_height


100%|██████████| 65/65 [04:40<00:00,  4.32s/it]

Question: What is the command to install the latest version of Optimum with OpenVINO support?

Answer: The command to install the latest version of Optimum with OpenVINO support is:

```bash
pip install --upgrade-strategy eager optimum["openvino"]
```

This command ensures `optimum-intel` is using the latest version.
True answer: pip install --upgrade-strategy eager optimum["openvino"]


In [ ]:
# agentic RAG

outputs_agentic_rag = []

for example in tqdm(eval_dataset):
    question = example["question"]

    enhanced_question = f"""Using the information contained in your knowledge base, which you can access with the 'retriever' tool,
give a comprehensive answer to the question below.
Respond only to the question asked, response should be concise and relevant to the question.
If you cannot find information, do not give up and try calling your retriever again with different arguments!
Make sure to have covered the question completely by calling the retriever tool several times with semantically different queries.
Your queries should not be questions but affirmative form sentences: e.g. rather than "How do I load a model from the Hub in bf16?", query should be "load a model from the Hub bf16 weights".

Question:
{question}"""
    answer = agent.run(enhanced_question)
    print("=======================================================")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print(f"True answer: {example['answer']}")

    results_agentic = {
        "question": question,
        "true_answer": example["answer"],
        "source_doc": example["source_doc"],
        "generated_answer": answer,
    }
    outputs_agentic_rag.append(results_agentic)

  0%|          | 0/65 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Using the information contained in your knowledge base, which you can access with the 'retriever' tool,         │
│ give a comprehensive answer to the question below.                                                              │
│ Respond only to the question asked, response should be concise and relevant to the question.                    │
│ If you cannot find information, do not give up and try calling your retriever again with different arguments!   │
│ Make sure to have covered the question completely by calling the retriever tool several times with semantically │
│ different queries.                                                                                              │
│ Your queries should not be questions but affirmative form sentences: e.g. rather than "How do I load a model    │
│ from the Hub in bf16?", query should be "load a model from the Hub bf16 weights".                               │
│                                                                                                                 │
│ Question:                                                                                                       │
│ What architecture is the `tokenizers-linux-x64-musl` binary designed for?                                       │
│                                                                                                                 │
╰─ HfApiModel - meta-llama/Llama-3.1-8B-Instruct ─────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'retriever' with arguments: {'query': '{"thedynamics binary\\} architecture'}                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Retrieved documents:
===== Document 0 =====
<img src="https://user-images.githubusercontent.com/11435359/146857310-f258c86c-fde6-48e8-9cee-badd2b21bd2c.png"
alt="drawing" width="600"/> 

<small> MAE architecture. Taken from the <a href="https://arxiv.org/abs/2111.06377">original paper.</a> 
</small>===== Document 1 =====
The model was trained on |Jean Zay](http://www.idris.fr/eng/jean-zay/jean-zay-presentation-eng.html), the French 
government-funded super computer that is managed by GENCI and installed at |IDRIS](http://www.idris.fr/), the 
national computing center for the French National Center for Scientific Research (CNRS). The compute was generously
donated to the project by GENCI (grant 2021-A0101012475).

The following hardware was used during the training:===== Document 2 =====
Therefore, the architecture can also naturally benefit from classic acceleration and compression techniques (such 
as |ONNX](https://github.com/harrisonvanderbyl/rwkv-onnx), 4-bit/8-bit quantization, etc.), and we hope this will 
be democratized for developers and practitioners together with the transformers integration of the 
architecture.===== Document 3 =====
Officially supported model architectures are currently:===== Document 4 =====
Supported architectures from |🤗 Transformers](https://huggingface.co/docs/transformers/index):===== Document 5 
=====
.html) architecture at zero cost.===== Document 6 =====
Supported architectures:

- Albert
- BERT
- Camembert
- ConvBert
- Deberta
- Deberta V2
- DistilBert
- Electra
- Flaubert
- MobileBert
- MPNet
- ResNet
- Roberta
- RoFormer
- XLM
- XLMRoberta

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'pip install --upgrade-strategy eager                   │
│ optimum[openvino]'}                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final answer: pip install --upgrade-strategy eager optimum[openvino]

[Step 2: Duration 2.32 seconds| Input tokens: 3,714 | Output tokens: 57]

100%|██████████| 65/65 [06:19<00:00,  5.83s/it]

Question: What is the command to install the latest version of Optimum with OpenVINO support?

Answer: pip install --upgrade-strategy eager optimum[openvino]
True answer: pip install --upgrade-strategy eager optimum["openvino"]


Ten kod implementuje agentowy potok Retrieval-Augmented Generation (RAG), który wykorzystuje agenta do iteracyjnego pobierania informacji i generowania odpowiedzi na pytania. Różni się od poprzedniego kodu tym, że używa agenta do bardziej aktywnego zarządzania procesem wyszukiwania kontekstu.

*   **`outputs_agentic_rag = []`:** Inicjalizuje pustą listę o nazwie `outputs_agentic_rag`. Ta lista będzie przechowywać wyniki dla każdego przykładu w zbiorze danych ewaluacyjnych, podobnie jak w poprzednim kodzie.

*   **`for example in tqdm(eval_dataset):`:** Rozpoczyna pętlę iterującą po każdym przykładzie (`example`) w zbiorze danych `eval_dataset`, z paskiem postępu wyświetlanym przez `tqdm`.

*   **`question = example["question"]`:** Pobiera pytanie z bieżącego przykładu i przypisuje je do zmiennej `question`.

*   **`enhanced_question = f"""..."""`:** Tworzy rozszerzony prompt dla agenta. Ten prompt różni się od poprzedniego tym, że zawiera instrukcje dotyczące:
    *   Wykorzystania narzędzia "retriever" do dostępu do bazy wiedzy.
    *   Iteracyjnego pobierania informacji poprzez wielokrotne wywoływanie narzędzia "retriever" z semantycznie różnymi zapytaniami.
    *   Formułowania zapytań w formie zdań oznajmujących, a nie pytań (np. "load a model from the Hub bf16 weights" zamiast "How do I load a model from the Hub in bf16?").  Ma to na celu poprawę skuteczności wyszukiwania.

*   **`answer = agent.run(enhanced_question)`:** Wywołuje metodę `run` agenta (`agent`), przekazując mu rozszerzony prompt. Agent samodzielnie decyduje, kiedy i jak wywołać narzędzie "retriever" w celu pobrania kontekstu, a następnie generuje odpowiedź na podstawie zebranych informacji.

*   **`print("=======================================================") ...`:** Wyświetla pytanie, wygenerowaną odpowiedź oraz poprawną odpowiedź dla każdego przykładu, służąc do wizualnej inspekcji wyników.

*   **`results_agentic = { ... }`:** Tworzy słownik o nazwie `results_agentic`, który zawiera:
    *   Pytanie (`question`).
    *   Poprawną odpowiedź (`example["answer"]`).
    *   Źródło dokumentu (`example["source_doc"]`).
    *   Wygenerowaną odpowiedź (`answer`).

*   **`outputs_agentic_rag.append(results_agentic)`:** Dodaje słownik `results_agentic` do listy `outputs_agentic_rag`.

W skrócie, ten kod implementuje agentowy potok RAG, w którym agent aktywnie zarządza procesem wyszukiwania kontekstu poprzez iteracyjne wywoływanie narzędzia "retriever" z różnymi zapytaniami. Pozwala to na bardziej elastyczne i skuteczne pobieranie informacji, co może prowadzić do lepszych odpowiedzi.

In [17]:
# Compare the two


In [ ]:
ii = 0

print("question: ", outputs_standard_rag[ii]["question"])
print("true answer: ", outputs_standard_rag[ii]["true_answer"])
print("ordinary RAG: ", outputs_standard_rag[ii]["generated_answer"])
print("agentic RAG: ", outputs_agentic_rag[ii]["generated_answer"])

question:  What architecture is the `tokenizers-linux-x64-musl` binary designed for?

true answer:  x86_64-unknown-linux-musl
ordinary RAG:  The `tokenizers-linux-x64-musl` binary is designed for aarch64-unknown-linux-musl architecture. 

Reference: Document 1
agentic RAG:  MobileBERT


In [ ]:
ii = 12

print("question: ", outputs_standard_rag[ii]["question"])
print("true answer: ", outputs_standard_rag[ii]["true_answer"])
print("ordinary RAG: ", outputs_standard_rag[ii]["generated_answer"])
print("agentic RAG: ", outputs_agentic_rag[ii]["generated_answer"])

question:  What task does the `roberta-large-mnli` checkpoint perform?

true answer:  Text classification
ordinary RAG:  The task that the `roberta-large-mnli` checkpoint performs is Sequence Classification, specifically for the Multi-Label Classification task (MNLI).
agentic RAG:  The RoBERTa-large-mnli checkpoint performs natural language inference.


In [ ]:
ii = 33


print("question: ", outputs_standard_rag[ii]["question"])
print("true answer: ", outputs_standard_rag[ii]["true_answer"])
print("ordinary RAG: ", outputs_standard_rag[ii]["generated_answer"])
print("agentic RAG: ", outputs_agentic_rag[ii]["generated_answer"])

question:  What library does MarkupLMFeatureExtractor use to extract data from HTML and XML files?

true answer:  Beautiful Soup
ordinary RAG:  MarkupLMFeatureExtractor uses Beautiful Soup, a Python library for pulling data out of HTML and XML files, under the hood (Source: Document 0).
agentic RAG:  Beautiful Soup
